# Leitura de dados de texto com colunas de largura fixa

## Exemplo aplicado: microdados da PNAD/IBGE

Em muitas bases públicas brasileiras, especialmente bases antigas ou microdados disponibilizados por órgãos oficiais, os dados não vêm organizados em arquivos `.csv` com separadores como vírgula, ponto e vírgula ou tabulação.

Um caso clássico é o dos microdados da **PNAD/IBGE**, em que cada linha do arquivo de texto representa uma unidade de observação — por exemplo, uma pessoa entrevistada —, mas as variáveis estão armazenadas em **posições fixas dentro da linha**.

Isso significa que o arquivo não possui nomes de colunas nem delimitadores visíveis. A identificação de cada variável depende de um **dicionário de layout**, que informa em quais posições da linha cada informação começa e termina.

## O problema

Em um arquivo `.csv`, normalmente conseguimos identificar as colunas porque existe um separador entre elas:

```text
ano;uf;sexo;idade;cor;peso
2014;25;2;34;8;12345
```

Já em um arquivo de largura fixa, as informações aparecem como uma sequência contínua de caracteres:

```text
201425...2...........034...8........................................
```

Para o computador, essa linha é apenas uma sequência de caracteres. Para transformá-la em uma tabela analisável, precisamos informar explicitamente quais intervalos de caracteres correspondem a cada variável.

## O que é um arquivo de largura fixa?

Um arquivo de largura fixa, ou **fixed-width file**, é um arquivo de texto no qual cada variável ocupa sempre o mesmo intervalo de posições em cada linha.

Por exemplo:

| Variável | Posição inicial | Posição final | Tamanho |
|---|---:|---:|---:|
| Ano | 1 | 4 | 4 |
| UF | 5 | 6 | 2 |
| Sexo | 18 | 18 | 1 |
| Idade | 27 | 29 | 3 |
| Cor/Raça | 33 | 33 | 1 |

Nesse tipo de arquivo, a separação das colunas não é feita por vírgulas ou ponto e vírgula, mas sim pela posição ocupada por cada variável dentro da linha.

## Objetivo do exemplo

Neste exemplo, vamos usar a função `read_fwf()` do `pandas` para ler um arquivo de microdados em formato de largura fixa.

A lógica da leitura é:

1. definir os nomes das variáveis que queremos extrair;
2. informar as posições inicial e final de cada variável na linha;
3. ler o arquivo de texto com `read_fwf()`;
4. recodificar algumas variáveis categóricas, como sexo e cor/raça;
5. utilizar os dados estruturados para análise.

In [1]:
from pandas import read_fwf
import pandas as pd

## Definição das variáveis de interesse

Primeiro, definimos os nomes das variáveis que serão extraídas do arquivo de texto.

Neste exemplo didático, vamos trabalhar com um conjunto reduzido de variáveis da PNAD:

- `ano`: ano da pesquisa;
- `uf`: unidade da federação;
- `sexo`: sexo da pessoa;
- `idade`: idade da pessoa;
- `cor`: cor ou raça;
- `peso`: peso amostral.

O peso amostral é importante porque os microdados da PNAD são provenientes de uma amostra. Para produzir estimativas populacionais, os registros precisam ser ponderados.

In [2]:
nomes = ['ano', 'uf', 'sexo', 'idade', 'cor', 'peso']

nomes

['ano', 'uf', 'sexo', 'idade', 'cor', 'peso']

## Definição das posições das variáveis

Agora, informamos ao Python em que posição da linha cada variável está localizada.

A função `read_fwf()` utiliza o argumento `colspecs`, que recebe uma lista de tuplas no formato:

```python
(posição_inicial, posição_final)
```

Importante: em Python, a contagem das posições começa em **zero**. Por isso, se o dicionário da PNAD informa que uma variável começa na posição 1, no Python ela deve ser informada como posição 0.

In [3]:
posicoes = [
    (0, 4),      # ano
    (4, 6),      # unidade da federação
    (17, 18),    # sexo
    (26, 29),    # idade
    (32, 33),    # cor/raça
    (767, 772)   # peso amostral
]

posicoes

[(0, 4), (4, 6), (17, 18), (26, 29), (32, 33), (767, 772)]

## Leitura do arquivo de microdados

A seguir, indicamos o caminho do arquivo `.txt` da PNAD e fazemos a leitura com `read_fwf()`.

O argumento `colspecs` informa as posições das variáveis e o argumento `names` informa os nomes das colunas que serão criadas no `DataFrame`.

Ajuste o caminho do arquivo de acordo com a pasta onde os microdados estão salvos no seu computador.

In [4]:
# Caminho do arquivo de microdados da PNAD
# Ajuste este caminho conforme a localização do arquivo no seu computador
file = '~/Data/IBGE/pnad_anual_2014/PES2014.txt'

df = read_fwf(
    file,
    colspecs=posicoes,
    names=nomes
)

df.head()

,ano,uf,sexo,idade,cor,peso
0,2014,25,4,86,8,623
1,2014,25,4,46,8,623
2,2014,25,2,13,8,623
3,2014,25,2,10,2,622
4,2014,25,4,81,8,623


## Recodificação de variáveis categóricas

Após a leitura, algumas variáveis ainda aparecem codificadas numericamente.

Por exemplo, a variável `sexo` pode aparecer como:

- `2`: masculino;
- `4`: feminino.

A variável `cor` também aparece codificada, conforme o dicionário da pesquisa. Para tornar a base mais legível, podemos substituir os códigos pelos respectivos rótulos.

In [5]:
df = (
    df.assign(
        sexo=lambda x: x.sexo.map({
            2: 'M',
            4: 'F'
        }),
        cor=lambda x: x.cor.map({
            0: 'Indígena',
            2: 'Branca',
            4: 'Preta',
            6: 'Amarela',
            8: 'Parda'
        })
    )
)

df.head()

,ano,uf,sexo,idade,cor,peso
0,2014,25,F,86,Parda,623
1,2014,25,F,46,Parda,623
2,2014,25,M,13,Parda,623
3,2014,25,M,10,Branca,622
4,2014,25,F,81,Parda,623


## Conferência inicial da base

Depois de carregar os dados, é recomendável fazer algumas verificações simples:

- número de linhas e colunas;
- tipos das variáveis;
- valores ausentes;
- categorias observadas nas variáveis qualitativas;
- valores mínimos e máximos das variáveis numéricas.

Essa etapa ajuda a identificar erros de leitura, posições incorretas ou problemas no layout.

In [6]:
df.shape

(6339, 6)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6339 entries, 0 to 6338
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ano     6339 non-null   int64 
 1   uf      6339 non-null   int64 
 2   sexo    6339 non-null   object
 3   idade   6339 non-null   int64 
 4   cor     6339 non-null   object
 5   peso    6339 non-null   int64 
dtypes: int64(4), object(2)
memory usage: 297.3+ KB


In [8]:
df[['sexo', 'cor']].value_counts(dropna=False)

sexo  cor     
F     Parda       1847
M     Parda       1711
F     Branca      1202
M     Branca      1093
      Preta        217
F     Preta        206
M     Indígena      30
F     Indígena      26
      Amarela        6
M     Amarela        1
Name: count, dtype: int64

## Exemplo de estatística descritiva simples

Com os dados estruturados em formato tabular, podemos calcular estatísticas descritivas.

Por exemplo, podemos observar a distribuição da população amostrada por sexo e cor/raça.

In [9]:
tab_sexo_cor = (
    df.groupby(['sexo', 'cor'], dropna=False)
      .size()
      .reset_index(name='n')
      .sort_values('n', ascending=False)
)

tab_sexo_cor

,sexo,cor,n
3,F,Parda,1847
8,M,Parda,1711
1,F,Branca,1202
6,M,Branca,1093
9,M,Preta,217
4,F,Preta,206
7,M,Indígena,30
2,F,Indígena,26
0,F,Amarela,6
5,M,Amarela,1


## Estatística ponderada pelo peso amostral

Em pesquisas amostrais, como a PNAD, cada registro representa um número de pessoas da população.

Por isso, uma tabulação simples de frequências mostra apenas a distribuição da amostra. Para estimar totais populacionais, devemos usar o peso amostral.

Abaixo, agregamos o peso por sexo e cor/raça.

In [10]:
tab_ponderada = (
    df.groupby(['sexo', 'cor'], dropna=False)
      .agg(populacao_estimada=('peso', 'sum'))
      .reset_index()
      .sort_values('populacao_estimada', ascending=False)
)

tab_ponderada

,sexo,cor,populacao_estimada
3,F,Parda,1150348
8,M,Parda,1065653
1,F,Branca,748624
6,M,Branca,680736
9,M,Preta,135143
4,F,Preta,128302
7,M,Indígena,18678
2,F,Indígena,16193
0,F,Amarela,3737
5,M,Amarela,623


# Leitura automática a partir do dicionário de variáveis

A definição manual das posições funciona bem para poucos campos, mas se torna trabalhosa e sujeita a erro quando queremos ler muitas variáveis.

Uma alternativa mais robusta é ler o próprio dicionário da PNAD e construir automaticamente as posições de cada variável.

Essa estratégia é especialmente útil em projetos maiores, nos quais precisamos carregar dezenas ou centenas de variáveis.

In [11]:
from pandas import read_excel

## Leitura do dicionário

O dicionário de variáveis geralmente informa:

- nome ou código da variável;
- descrição da variável;
- posição inicial;
- tamanho do campo;
- categorias ou códigos válidos.

No exemplo abaixo, o arquivo do dicionário está em Excel. Ajuste o caminho conforme a localização do arquivo no seu computador.

In [13]:
# Caminho do dicionário de variáveis
# Ajuste este caminho conforme a localização do arquivo no seu computador
dic_file = '~/Data/IBGE/pnad_anual_2014/Dicionario de variaveis de pessoas - PNAD 2014.xlsx'

dic = read_excel(
    dic_file,
    skiprows=1
)

dic.head()

,Posição Inicial,Tamanho,Código de variável,Quesito,Unnamed: 4,Categorias,Unnamed: 6
0,NaN,NaN,NaN,N°,Descrição,Tipo,Descrição
1,PESQUISA BÁSICA,NaN,NaN,NaN,NaN,NaN,NaN
2,PARTE 1 – IDENTIFICAÇÃO E CONTROLE,NaN,NaN,NaN,NaN,NaN,NaN
3,1,4.0,V0101,NaN,Ano de referência,NaN,NaN
4,5,2.0,UF,2,Unidade da Federação,11,Rondônia


## Construção automática das posições

A partir das colunas `Posição Inicial` e `Tamanho`, criamos duas novas colunas:

- `inicio`: posição inicial ajustada para a indexação do Python, que começa em zero;
- `final`: posição final da variável.

Depois, transformamos essas posições em uma lista de tuplas para usar no argumento `colspecs` da função `read_fwf()`.

In [14]:
dic_layout = (
    dic[['Posição Inicial', 'Tamanho', 'Código de variável']]
    .dropna(subset='Código de variável')
    .assign(
        inicio=lambda x: x['Posição Inicial'] - 1,
        final=lambda x: x.inicio + x['Tamanho']
    )
)

dic_layout.head()

,Posição Inicial,Tamanho,Código de variável,inicio,final
3,1,4.0,V0101,0,4.0
4,5,2.0,UF,4,6.0
31,5,8.0,V0102,4,12.0
32,13,3.0,V0103,12,15.0
34,16,2.0,V0301,15,17.0


In [16]:
nomes_auto = dic_layout['Código de variável'].to_list()

posicoes_auto = list(
    zip(
        dic_layout.inicio.astype(int),
        dic_layout.final.astype(int)
    )
)

nomes_auto[:10], posicoes_auto[:10]

(['V0101',
  'UF',
  'V0102',
  'V0103',
  'V0301',
  'V0302',
  'V3031',
  'V3032',
  'V3033',
  'V8005'],
 [(0, 4),
  (4, 6),
  (4, 12),
  (12, 15),
  (15, 17),
  (17, 18),
  (18, 20),
  (20, 22),
  (22, 26),
  (26, 29)])

## Leitura com layout automático

Agora, usamos as posições extraídas automaticamente do dicionário para ler o arquivo de microdados.

No exemplo abaixo, usamos `nrows=10` apenas para testar rapidamente a leitura. Em uma aplicação real, esse argumento pode ser removido para carregar a base completa.

In [17]:
df_auto = read_fwf(
    file,
    colspecs=posicoes_auto,
    names=nomes_auto,
    nrows=10
)

df_auto.head()

,V0101,UF,V0102,V0103,V0301,V0302,V3031,V3032,V3033,V8005,...,V4741,V4742,V4743,V4745,V4746,V4747,V4748,V4749,V4750,V9993
0,2014,25,25000012,1,1,4,5,2,1928,86,...,4,462,3,1,2,NaN,NaN,2,462,20160623
1,2014,25,25000012,1,2,4,28,11,1967,46,...,4,462,3,2,1,2.0,2.0,1,462,20160623
2,2014,25,25000012,1,3,2,4,12,2000,13,...,4,462,3,2,2,NaN,NaN,2,462,20160623
3,2014,25,25000012,1,4,2,5,6,2004,10,...,4,462,3,1,2,NaN,NaN,2,462,20160623
4,2014,25,25000012,2,1,4,31,5,1933,81,...,2,1086,4,1,2,NaN,NaN,2,1086,20160623


## Vantagens da leitura automática pelo dicionário

A leitura automática a partir do dicionário apresenta algumas vantagens:

1. reduz o risco de erro manual nas posições;
2. facilita a leitura de muitas variáveis;
3. torna o código mais reprodutível;
4. facilita a adaptação para outros anos da pesquisa;
5. aproxima o processo de leitura da documentação oficial da base.

Essa abordagem é mais indicada quando se trabalha com bases grandes, projetos recorrentes ou processos de ETL.

# Exercício sugerido

Usando o arquivo da PNAD e o dicionário de variáveis, faça as seguintes tarefas:

1. escolha cinco variáveis adicionais no dicionário;
2. identifique suas posições iniciais e seus tamanhos;
3. leia essas variáveis usando `read_fwf()`;
4. recodifique pelo menos uma variável categórica;
5. produza uma tabela descritiva usando o peso amostral.

Pergunta orientadora:

> Como a estrutura da população varia por sexo, cor/raça e unidade da federação quando usamos os pesos amostrais da PNAD?